In [2]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [9]:
import sklearn


In [11]:
# --- Environment Setup for VS Code ---
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk

# Download NLTK stopwords (only the first time)
nltk.download('stopwords')
from nltk.corpus import stopwords
stopwords_pt = stopwords.words('portuguese')




[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marga\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#Sprint 2

In [24]:
# carregar o dataset 

parquet_path = "../../data_sample/cision_news_rows.parquet"

sample = pd.read_parquet(parquet_path, engine="pyarrow")

sample.head()


,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria
0,117766326,Novo regime de arrendamento com opção de compr...,O novo regime de arrendamento com opção de com...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1105.4,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
1,117766609,TRIBUNAL DETECTA OBRAS IRREGULARES EM SANTANA,Tribunal detecta obras irregulares em Santana ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,Miguel Fernandes Luís,1323.1,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
2,117766528,Relatório final da Comissão Parlamentar de Inq...,“Os processos de contratualização subjacente à...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1097.6,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
3,117766532,Valença prevê concluir em 2026 obras no Centro...,A reabilitação e ampliação do Centro de Saúde ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
4,117766576,CHUMBADA MOÇÃO DE REJEIÇÃO DO PCP,Após a votação à moção de rejeição do PCP ao P...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1445.2,http://pt.cision.com/cp2013/clippingdetails.as...,NaN


In [25]:
sample.head(-100)

,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria
0,117766326,Novo regime de arrendamento com opção de compr...,O novo regime de arrendamento com opção de com...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1105.40,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
1,117766609,TRIBUNAL DETECTA OBRAS IRREGULARES EM SANTANA,Tribunal detecta obras irregulares em Santana ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,Miguel Fernandes Luís,1323.10,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
2,117766528,Relatório final da Comissão Parlamentar de Inq...,“Os processos de contratualização subjacente à...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1097.60,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
3,117766532,Valença prevê concluir em 2026 obras no Centro...,A reabilitação e ampliação do Centro de Saúde ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.00,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
4,117766576,CHUMBADA MOÇÃO DE REJEIÇÃO DO PCP,Após a votação à moção de rejeição do PCP ao P...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1445.20,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
...,...,...,...,...,...,...,...,...,...,...
19895,119165457,PS apresentou Rui Miguel Cruz para a presidênc...,O PS apresentou Rui Miguel Cruz como candidato...,https://centralpress.pt/page/107604/redacao/20...,2025-09-16 00:00:00+00:00,Internet,Samuel Guiomar,111.82,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
19896,119165646,Carta aberta une personalidades portuguesas à ...,"Dezenas de portugueses, numa carta aberta, anu...",https://bomdia.eu/carta-aberta-une-personalida...,2025-09-16 00:00:00+00:00,Internet,Fabiana Bravo,600.00,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
19897,119165554,Ventura aponta a Gouveia e Melo com risco e se...,"Tal como parecia inevitável há muito, e foi av...",https://www.dn.pt/pol%C3%ADtica/ventura-aponta...,2025-09-16 00:00:00+00:00,Internet,Leonardo Ralha,16937.38,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
19898,119165563,Governo disponibiliza 30 milhões de euros para...,O Governo vai disponibilizar 30 milhões de eur...,https://odigital.sapo.pt/governo-disponibiliza...,2025-09-16 00:00:00+00:00,Internet,sem autor,272.44,http://pt.cision.com/cp2013/clippingdetails.as...,NaN


In [26]:
print(sample.shape)
print(sample.columns)
display(sample.head())
display(sample.dtypes)
print(sample.isna().sum())
print("Duplicados por corpo de noticia:", sample.duplicated(subset=["noticia"]).sum()) #mas têm valores unitarios dif


(20000, 10)
Index(['id', 'titulo', 'noticia', 'link', 'data_publicacao', 'tipo_meio',
       'autores', 'valor_publicitario', 'guid', 'categoria'],
      dtype='object')


,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria
0,117766326,Novo regime de arrendamento com opção de compr...,O novo regime de arrendamento com opção de com...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1105.4,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
1,117766609,TRIBUNAL DETECTA OBRAS IRREGULARES EM SANTANA,Tribunal detecta obras irregulares em Santana ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,Miguel Fernandes Luís,1323.1,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
2,117766528,Relatório final da Comissão Parlamentar de Inq...,“Os processos de contratualização subjacente à...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1097.6,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
3,117766532,Valença prevê concluir em 2026 obras no Centro...,A reabilitação e ampliação do Centro de Saúde ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
4,117766576,CHUMBADA MOÇÃO DE REJEIÇÃO DO PCP,Após a votação à moção de rejeição do PCP ao P...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1445.2,http://pt.cision.com/cp2013/clippingdetails.as...,NaN


id                                  int64
titulo                             object
noticia                            object
link                               object
data_publicacao       datetime64[ns, UTC]
tipo_meio                          object
autores                            object
valor_publicitario                float64
guid                               object
categoria                         float64
dtype: object

id                        0
titulo                    0
noticia                   0
link                      0
data_publicacao           0
tipo_meio                 0
autores                   0
valor_publicitario        0
guid                      0
categoria             20000
dtype: int64
Duplicados por corpo de noticia: 201


só a coluna "categoria" é que tem nulos

In [27]:
# Datas e valores numéricos
sample["data_publicacao"] = pd.to_datetime(sample["data_publicacao"], errors="coerce")
sample["valor_publicitario"] = pd.to_numeric(sample["valor_publicitario"], errors="coerce")

# Normalizar texto das colunas relevantes
def norm_txt(s):
    s = str(s).lower()
    s = re.sub(r"[\:\;\,\.\!\?\(\)\[\]\-\–\—\"\']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

#vamos ver com o corpo da noticia
sample["noticia"] = sample["noticia"].map(norm_txt)


##TF-IDF (melhor opção de todas)

In [28]:
#4. Vetorização TF-IDF + matriz de similaridade
vectorizer = TfidfVectorizer(stop_words=list(stopwords_pt), ngram_range=(1,2), max_features=20000, min_df=2)
X = vectorizer.fit_transform(sample["noticia"])

# Similaridade (mais baixo aumenta grupos)
S = cosine_similarity(X, dense_output=False)
threshold = 0.76

In [29]:
#5. Agrupamento por similaridade
n = S.shape[0]
visitados = np.zeros(n, dtype=bool)
grupos = []
for i in range(n):
    if visitados[i]:
        continue
    similares = S[i].toarray().ravel() >= threshold
    grupo_idxs = np.where(similares)[0].tolist()
    visitados[grupo_idxs] = True
    grupos.append(grupo_idxs)

grupo_id_map = {idx: gid for gid, g in enumerate(grupos) for idx in g}
sample["grupo_id"] = sample.index.map(grupo_id_map.get).astype("Int64")


In [30]:
# para contagens
from collections import Counter
sizes = Counter(len(g) for g in grupos)
print('Distribuição de tamanhos de grupo:', sizes)
print('Grupos totais:', len(grupos))
print('Grupos com 1 notícia:', sum(1 for g in grupos if len(g)==1))


Distribuição de tamanhos de grupo: Counter({1: 12523, 2: 1579, 3: 413, 4: 185, 5: 107, 6: 62, 7: 35, 10: 26, 8: 18, 9: 17, 12: 10, 13: 8, 14: 7, 17: 6, 19: 6, 11: 6, 20: 6, 15: 4, 16: 3, 23: 3, 25: 3, 18: 2, 26: 1, 31: 1, 27: 1, 24: 1})
Grupos totais: 15033
Grupos com 1 notícia: 12523


In [31]:
# Tabela com número de notícias por grupo
contagem_por_grupo = (
    sample.groupby("grupo_id")
          .size()
          .reset_index(name="num_noticias")
)

# Lista com os ids dos grupos que têm exatamente 32 notícias p.e.
grupos_x = contagem_por_grupo.loc[
    contagem_por_grupo["num_noticias"] == 15, "grupo_id"
].tolist()

print("Grupos com x notícias:", grupos_x)
print("Total de grupos com x notícias:", len(grupos_x))

Grupos com x notícias: [1513, 4556, 4576, 12149]
Total de grupos com x notícias: 4


In [32]:
#6. Escolher o líder por valor publicitário

def lider_valor_publicitario(df, grupo, col_valor="valor_publicitario"):
    # apenas retorna o índice da notícia com maior valor_publicitario dentro do grupo
    idx_validos = [int(i) for i in grupo if 0 <= int(i) < len(df)]
    return int(df.loc[idx_validos, col_valor].idxmax())

lideres = [lider_valor_publicitario(sample, g) for g in grupos]
sample["is_lider"] = False
lideres_validos = [i for i in lideres if i is not None and 0 <= int(i) < len(sample)]
sample.loc[lideres_validos, "is_lider"] = True


In [33]:
sample.head(60)

#aqui da para ver que o 47 e o 26 sao exatamene iguais mas por terem sido publicados a uma hora dif o 47 tem valor € mt superior

,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria,grupo_id,is_lider
0,117766326,Novo regime de arrendamento com opção de compr...,o novo regime de arrendamento com opção de com...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1105.4,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,0,False
1,117766609,TRIBUNAL DETECTA OBRAS IRREGULARES EM SANTANA,tribunal detecta obras irregulares em santana ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,Miguel Fernandes Luís,1323.1,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,1,True
2,117766528,Relatório final da Comissão Parlamentar de Inq...,“os processos de contratualização subjacente à...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1097.6,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,2,True
3,117766532,Valença prevê concluir em 2026 obras no Centro...,a reabilitação e ampliação do centro de saúde ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,3,True
4,117766576,CHUMBADA MOÇÃO DE REJEIÇÃO DO PCP,após a votação à moção de rejeição do pcp ao p...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1445.2,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,4,True
5,117750707,Investimento superior a 2 milhões de euros - P...,oinstituto politécnico de leiria lançou o conc...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,5,True
6,117750710,34.º Aniversário da elevação de Pombal a cidad...,perceber como era pombal há 34 anos e ver a ci...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,314.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,6,True
7,117768156,Comissão decide instaurar ação contra Portugal...,curiosidades comissão decide instaurar ação co...,https://terrasdohomem.pt/2025/06/19/comissao-d...,2025-06-19 00:00:00+00:00,Internet,sem autor,102.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,7,True
8,117768279,Investigadores desenvolvem base de dados e mod...,uma equipa de investigadores da faculdade de c...,https://guimaraesagora.pt/investigadores-desen...,2025-06-19 00:00:00+00:00,Internet,sem autor,114.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,8,True
9,117749968,Visita às obras - Requalificação do Parque de ...,o presidente da câmara municipal paulo ferreir...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN,9,True


In [34]:
#dos 4 grupos que tem 15 noticias
gid = grupos_x[0] #0 é o 1 grupo, 1 o segundo e etc
exemplo_4 = sample.loc[sample["grupo_id"] == gid, ["grupo_id","is_lider","noticia","data_publicacao", "valor_publicitario"]].head(50)
display(exemplo_4)


,grupo_id,is_lider,noticia,data_publicacao,valor_publicitario
1823,1513,False,manifestação a líder parlamentar do livre o se...,2025-06-28 00:00:00+00:00,5927.14
1824,1513,True,durante a manifestação que decorreu entre o ch...,2025-06-28 00:00:00+00:00,44547.31
1827,1513,False,a líder parlamentar do livre o secretário gera...,2025-06-28 00:00:00+00:00,6417.82
1832,1513,False,leonardo negrão manifestantes exigem que o gov...,2025-06-28 00:00:00+00:00,18679.86
1833,1513,False,a líder parlamentar do livre o secretário gera...,2025-06-28 00:00:00+00:00,28767.94
1836,1513,False,a líder parlamentar do livre o secretário gera...,2025-06-28 00:00:00+00:00,43263.40
1847,1513,False,a líder parlamentar do livre o secretário gera...,2025-06-28 00:00:00+00:00,18541.85
1857,1513,False,a manifestação pelo direito à habitação juntou...,2025-06-28 00:00:00+00:00,16324.90
1894,1513,False,livre pcp e bloco acusam governo de agravar cr...,2025-06-28 00:00:00+00:00,418.29
1899,1513,False,casa é direito não luxo voltar às cooperativas...,2025-06-28 00:00:00+00:00,103.18


In [35]:
ver_carac = sample[sample["grupo_id"] == 6042]

display(ver_carac[["is_lider", "noticia", "tipo_meio", "valor_publicitario"]])


,is_lider,noticia,tipo_meio,valor_publicitario
7842,False,a empreitada de reabilitação do teatro clube d...,Internet,148.0
8629,False,foi inaugurada na quinta feira dia 17 de julho...,Internet,108.0
8787,True,com investimento superior a dois milhões e mei...,Imprensa,654.1


In [36]:
# Inspecionar um grupo grande
gid_ex = sample["grupo_id"].value_counts().idxmax()
display(sample[sample["grupo_id"] == gid_ex][["id", "is_lider", "noticia", "valor_publicitario"]].head(50))


,id,is_lider,noticia,valor_publicitario
6241,118161533,False,a presidente da comissão europeia anunciou est...,24699.0
6305,118164410,False,a presidente da comissão europeia anunciou est...,331.0
6308,118165694,False,foto epa política 13 jul 2025 18 55 ue suspend...,24699.0
6314,118165095,False,a presidente da comissão europeia anunciou que...,418.0
6354,118162323,False,i internacional ue suspende medidas retaliação...,218.0
6357,118162272,False,presidente da comissão europeia anunciou que v...,34730.0
6362,118162237,False,a presidente da comissão europeia anunciou hoj...,4281.0
6364,118162142,False,a presidente da comissão europeia anunciou est...,600.0
6366,118162031,False,ursula von der leyen continua a preparar as co...,16826.0
6370,118161791,False,von der leyen argumenta que apesar de manter a...,6949.0


## SBERT + kmeans (uma das opcões que testámos)

In [39]:

from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans


c:\Users\marga\Downloads\Planapp-main (1)\Planapp-main\Planapp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [40]:
parquet_path = "../../data_sample/cision_news_rows.parquet"
sample = pd.read_parquet(parquet_path, engine="pyarrow")

# Usa só o corpo da notícia
sample["noticia"] = sample["noticia"].astype(str).fillna("")

print(f" Dataset carregado com {len(sample)} linhas e {len(sample.columns)} colunas.")
sample.head()



 Dataset carregado com 20000 linhas e 10 colunas.


,id,titulo,noticia,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria
0,117766326,Novo regime de arrendamento com opção de compr...,O novo regime de arrendamento com opção de com...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1105.4,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
1,117766609,TRIBUNAL DETECTA OBRAS IRREGULARES EM SANTANA,Tribunal detecta obras irregulares em Santana ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,Miguel Fernandes Luís,1323.1,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
2,117766528,Relatório final da Comissão Parlamentar de Inq...,“Os processos de contratualização subjacente à...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1097.6,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
3,117766532,Valença prevê concluir em 2026 obras no Centro...,A reabilitação e ampliação do Centro de Saúde ...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,NaN
4,117766576,CHUMBADA MOÇÃO DE REJEIÇÃO DO PCP,Após a votação à moção de rejeição do PCP ao P...,No link available,2025-06-19 00:00:00+00:00,Imprensa,sem autor,1445.2,http://pt.cision.com/cp2013/clippingdetails.as...,NaN


In [18]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Pode demorar alguns minutos para 20k textos!
embeddings = model.encode(sample["noticia"].tolist(), batch_size=64, show_progress_bar=True)


c:\Users\inesc\Downloads\ISCTE\Planapp\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\inesc\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package i

In [19]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

# Lista dos valores de clusters que queres testar
cluster_tests = [200, 500, 700, 1000]

# Dicionário para guardar os resultados
silhouette_scores = {}

for n_clusters in cluster_tests:
    print(f"\n===== Testando n_clusters = {n_clusters} =====")
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=256, max_iter=500)
    labels = kmeans.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels)
    silhouette_scores[n_clusters] = score
    print(f"Silhouette Score: {score:.3f}")

    # Guarda clusters no DataFrame para poderes explorar depois
    sample[f"cluster_{n_clusters}"] = labels

    # Exemplo: distribuições dos tamanhos de cluster
    print("Distribuição dos tamanhos:", pd.Series(labels).value_counts().head(10).to_dict())



===== Testando n_clusters = 200 =====
Silhouette Score: -0.028
Distribuição dos tamanhos: {76: 702, 99: 539, 6: 409, 161: 403, 119: 376, 53: 363, 26: 358, 159: 337, 33: 301, 49: 289}

===== Testando n_clusters = 500 =====
Silhouette Score: -0.032
Distribuição dos tamanhos: {63: 360, 7: 330, 182: 289, 125: 268, 378: 238, 85: 234, 370: 226, 453: 218, 316: 207, 74: 205}

===== Testando n_clusters = 700 =====
Silhouette Score: -0.042
Distribuição dos tamanhos: {448: 286, 534: 264, 453: 251, 29: 224, 385: 219, 139: 216, 446: 212, 7: 203, 170: 203, 655: 200}

===== Testando n_clusters = 1000 =====
Silhouette Score: -0.037
Distribuição dos tamanhos: {217: 253, 26: 250, 708: 238, 787: 229, 394: 216, 317: 198, 95: 179, 460: 163, 296: 162, 442: 153}


In [20]:
cluster_n = 1000

# Ver quantidades por grupo
print(sample[f"cluster_{cluster_n}"].value_counts().sort_values(ascending=False).head(10))

# Ver detalhes de um cluster específico (ex: 5)
g = 5
display(sample[sample[f"cluster_{cluster_n}"] == g][["id", "noticia"]].head(10))

# Ver o líder por valor publicitário em cada cluster escolhido
sample["valor_publicitario"] = pd.to_numeric(sample["valor_publicitario"], errors="coerce")
sample["is_lider"] = (
    sample.groupby(f"cluster_{cluster_n}")["valor_publicitario"]
    .transform(lambda x: x == x.max())
)
display(sample[sample["is_lider"]][[f"cluster_{cluster_n}", "id", "noticia", "valor_publicitario"]].head(10))


cluster_1000
217    253
26     250
708    238
787    229
394    216
317    198
95     179
460    163
296    162
442    153
Name: count, dtype: int64


,id,noticia
17277,119007270,Trump garantiu que se a União Europeia (UE) nã...
17303,119006859,Trump garantiu que se a União Europeia (UE) nã...
17314,119007097,"""A Europa atacou hoje uma outra grande empresa..."
17467,119006577,"""A Europa atacou hoje uma outra grande empresa..."


,cluster_1000,id,noticia,valor_publicitario
31,88,117767526,Augusto Santos Silva diz que não abdica de qua...,12687.0
32,229,117767587,De Coimbra vai seguir para a Assembleia da Rep...,26179.0
45,23,117772108,Os juros da casa desceram em maio pelo 16º mês...,298822.0
51,968,117773379,O ministro das Finanças está no Luxemburgo par...,30752.0
80,898,117776710,"O ministro das Finanças, Joaquim Miranda Sarme...",477761.0
82,140,117776742,Gouveia e Melo criticou os últimos Governos pe...,399216.0
83,297,117776871,Esquerda/Direita com Luís Campos Ferreira e Eu...,395472.0
102,311,117804062,"Sangue, sémen e fezes transmitem micro-organis...",5534.0
116,997,117844375,EXPECTATIVA SOBRE A DECISÃO da localização do ...,309.1
152,465,117798916,Os governos de França e Itália defenderam que ...,28768.0
